# FleetIQ DMS Training: End-to-End Visual Walkthrough

This notebook reuses the production-shaped DMS training modules and exposes each stage: feature extraction, temporal batching, Bi-LSTM training, validation, and checkpoint inspection. Run cells from top to bottom.

## 1. Setup and configured device

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().parents[1]
DMS_SRC = REPO_ROOT / "ml" / "training" / "dms" / "src"
for path in (REPO_ROOT, DMS_SRC):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from fleetiq_training_dms.config import Config
from fleetiq_training_dms.dataset import FEATURE_COLS, get_temporal_block_dataloaders
from fleetiq_training_dms.feature_extractor import extract_all_and_save
from fleetiq_training_dms.model import build_sequence_model
from fleetiq_training_dms.train import train_epoch, validate_epoch

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
Config.DEVICE = DEVICE

print(f"Repository: {REPO_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Epochs from Config: {Config.EPOCHS}")
print(f"Sequence length: {Config.SEQ_LEN}")

## 2. Dataset readiness

In [ ]:
missing_trips = [
    trip_id for trip_id in Config.ALL_TRIPS
    if not (Config.DATA_ROOT / trip_id).is_dir()
]
if not Config.DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset root not found: {Config.DATA_ROOT}")
if missing_trips:
    raise FileNotFoundError(f"Missing configured trips under {Config.DATA_ROOT}: {missing_trips}")

trip_counts = {
    trip_id: len(list((Config.DATA_ROOT / trip_id / "driver").glob("*.jpg")))
    for trip_id in Config.ALL_TRIPS
}
display(pd.Series(trip_counts, name="driver_frames").to_frame())
print(f"Feature output directory: {Config.FEATURE_DIR}")
print(f"Checkpoint output: {Config.OUTPUT_DIR / 'best_sequence_model.pt'}")

## 3. Extract and inspect 18 features

The extractor uses MediaPipe landmarks, head pose, image brightness, motion, deltas, and rolling statistics. If the MediaPipe task asset is missing, the existing extractor downloads it to the configured model directory.

In [ ]:
extract_all_and_save()
feature_paths = [Config.FEATURE_DIR / f"{trip_id}_features.csv" for trip_id in Config.ALL_TRIPS]
features = pd.concat(
    [pd.read_csv(path).assign(trip_id=path.stem.removesuffix("_features")) for path in feature_paths if path.exists()],
    ignore_index=True,
)
print(f"Feature rows: {len(features):,}")
display(features.head())
display(features[FEATURE_COLS].describe().T.round(4))

In [ ]:
label_names = features["state_label"].map(Config.STATE_INV_MAP).fillna("unknown")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
label_names.value_counts().reindex([*Config.STATE_MAP, "unknown"], fill_value=0).plot.bar(ax=axes[0], color="#2563eb")
axes[0].set_title("Driver-state class balance")
axes[0].set_xlabel("State")
axes[0].set_ylabel("Frames")
features[["ear", "mar", "pitch", "yaw", "roll"]].plot(ax=axes[1], alpha=0.75)
axes[1].set_title("Example continuous features")
axes[1].set_xlabel("Frame row")
axes[1].legend(ncol=2)
plt.tight_layout()
plt.show()

## 4. Build temporal dataloaders

In [ ]:
train_loader, val_loader, mean_scaler, std_scaler = get_temporal_block_dataloaders(
    Config.FEATURE_DIR,
    Config.ALL_TRIPS,
    seq_len=Config.SEQ_LEN,
    batch_size=Config.BATCH_SIZE,
    train_ratio=0.8,
)
sample_x, sample_y, sample_frame_ids, sample_trip_ids = next(iter(train_loader))
print(f"Input batch shape: {tuple(sample_x.shape)} = [batch, sequence, features]")
print(f"Label batch shape: {tuple(sample_y.shape)}")
print(f"Feature count: {sample_x.shape[-1]}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample_x[0].numpy())
ax.set_title(f"One normalized temporal window ({sample_x.shape[1]} frames)")
ax.set_xlabel("Relative frame")
ax.set_ylabel("Normalized feature value")
ax.legend(FEATURE_COLS, ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Train the Bi-LSTM

This cell mirrors the existing training entry point while retaining per-epoch history for visualization. The epoch count comes directly from `Config.EPOCHS`.

In [ ]:
model = build_sequence_model(
    feature_dim=len(FEATURE_COLS),
    hidden_dim=Config.HIDDEN_DIM,
    num_layers=Config.NUM_LAYERS,
    num_classes=Config.NUM_CLASSES,
    cell_type=Config.MODEL_TYPE,
).to(DEVICE)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)
history = []
best_val_acc = 0.0
Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_path = Config.OUTPUT_DIR / "best_sequence_model.pt"

for epoch in range(1, Config.EPOCHS + 1):
    started = time.perf_counter()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()
    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc})
    print(f"Epoch [{epoch:02d}/{Config.EPOCHS:02d}] | train acc {train_acc:.4f} | val acc {val_acc:.4f} | {time.perf_counter() - started:.1f}s")
    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_acc": val_acc,
            "seq_len": Config.SEQ_LEN,
            "feature_dim": len(FEATURE_COLS),
            "mean_scaler": mean_scaler,
            "std_scaler": std_scaler,
        }, best_path)

history_df = pd.DataFrame(history)
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"Saved checkpoint: {best_path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
history_df.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], marker="o")
axes[0].set_title("Loss by epoch")
history_df.plot(x="epoch", y=["train_acc", "val_acc"], ax=axes[1], marker="o")
axes[1].set_title("Accuracy by epoch")
axes[1].set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 6. Validate and inspect predictions

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
actual, predicted = [], []
with torch.no_grad():
    for x, y, _, _ in val_loader:
        logits = model(x.to(DEVICE))
        actual.extend(y.numpy().tolist())
        predicted.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())

names = [Config.STATE_INV_MAP[i] for i in range(Config.NUM_CLASSES)]
print(classification_report(actual, predicted, labels=list(range(Config.NUM_CLASSES)), target_names=names, zero_division=0))
cm = confusion_matrix(actual, predicted, labels=list(range(Config.NUM_CLASSES)))
ConfusionMatrixDisplay(cm, display_labels=names).plot(xticks_rotation=35, cmap="Blues")
plt.title("DMS validation confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
preview = pd.DataFrame({
    "actual": [Config.STATE_INV_MAP.get(value, "unknown") for value in actual[:20]],
    "predicted": [Config.STATE_INV_MAP.get(value, "unknown") for value in predicted[:20]],
})
display(preview)
print("Notebook complete: features extracted, model trained, checkpoint saved, and validation visualized.")

## 7. Optional phone-use detection and visualization

`phone_use` is an independent signal from the broad driver-state class. The cells below can run the existing detector when YOLO is available, then visualize the smoothed result.

In [ ]:
from fleetiq_training_dms.phone_detector import PhoneUseDetector, PhoneUseSmoother

PHONE_TRIP_ID = "T01-Sample"
PHONE_MODEL = REPO_ROOT / "yolo11n.pt"
PHONE_CONFIDENCE = 0.40
phone_results = None
phone_trip_dir = Config.DATA_ROOT / PHONE_TRIP_ID
phone_frames = sorted((phone_trip_dir / "driver").glob("frame_*.jpg"))

if not phone_frames:
    print(f"No driver frames found under {phone_trip_dir}")
elif not PHONE_MODEL.is_file():
    print(f"Phone model not found: {PHONE_MODEL}")
    print("Prepare it with: uv run --with ultralytics python -c 'from ultralytics import YOLO; YOLO(\"yolo11n.pt\")'")
else:
    try:
        detector = PhoneUseDetector(PHONE_MODEL, confidence=PHONE_CONFIDENCE)
        smoother = PhoneUseSmoother()
        phone_rows = []
        for image_path in phone_frames:
            frame_id = int(image_path.stem.rsplit("_", 1)[-1])
            raw_phone_use = detector.detect(image_path)
            phone_rows.append({
                "frame_id": frame_id,
                "phone_use": smoother.update(raw_phone_use),
                "image_path": image_path,
            })
        phone_results = pd.DataFrame(phone_rows)
        print(f"Detected phone-use positives: {int(phone_results.phone_use.eq(True).sum())}")
    except Exception as exc:
        print(f"Phone detector unavailable: {exc}")
        print("Use the documented prediction command to create a CSV instead.")

In [ ]:
phone_csv = Config.PRED_DIR / f"{PHONE_TRIP_ID}_twostage.csv"
if phone_results is not None:
    phone_view = phone_results.copy()
elif phone_csv.is_file():
    phone_view = pd.read_csv(phone_csv)
    phone_view["image_path"] = phone_view["frame_id"].map(
        lambda frame_id: phone_trip_dir / "driver" / f"frame_{int(frame_id):06d}.jpg"
    )
else:
    phone_view = pd.DataFrame()
    print(f"No phone-use results found. Generate {phone_csv} with the DMS prediction command.")

if not phone_view.empty:
    phone_view["phone_use"] = phone_view["phone_use"].map({True: True, False: False, "True": True, "False": False})
    display(phone_view["phone_use"].value_counts(dropna=False).rename("frames").to_frame())
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.step(phone_view["frame_id"], phone_view["phone_use"].map({True: 1, False: 0}), where="post", color="#f97316")
    ax.set_title(f"{PHONE_TRIP_ID} phone-use timeline")
    ax.set_xlabel("Frame")
    ax.set_yticks([0, 1], ["Not detected", "Detected"])
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.show()

    examples = phone_view[phone_view["phone_use"] == True].head(3)
    if not examples.empty:
        fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 4), squeeze=False)
        for axis, (_, row) in zip(axes[0], examples.iterrows(), strict=False):
            axis.imshow(plt.imread(row["image_path"]))
            axis.set_title(f"Frame {int(row.frame_id)} — phone use")
            axis.axis("off")
        plt.tight_layout()
        plt.show()

## 8. Optional DMD generalization training

DMD must be downloaded separately after accepting its academic/non-commercial terms. Paste the official archive URL into the downloader command; the repository does not bypass the DMD portal.

In [ ]:
dmd_root = REPO_ROOT / "data" / "DMD"
dmd_manifest = REPO_ROOT / "artifacts" / "training" / "dms" / "dmd_manifest.csv"
print("Browse supported datasets and official access pages:")
print("python tools/dataset/download_driver_datasets.py --list")
print("Download after accepting a dataset's terms:")
print("python tools/dataset/download_driver_datasets.py --dataset yawdd --url yawdd=<official-archive-url>")

if not dmd_root.is_dir():
    print(f"DMD is not installed yet: {dmd_root}")
elif not dmd_manifest.is_file():
    print(f"DMD found, but manifest is missing: {dmd_manifest}")
    print("Create a manifest with image_path, subject_id, and label columns before training.")
else:
    dmd_records = pd.read_csv(dmd_manifest)
    display(dmd_records.groupby("label")["subject_id"].agg(["count", "nunique"]))
    print("Subject-held-out training command:")
    print("uv run --package fleetiq-training-dms fleetiq-train-dms-generalized \
    --manifest artifacts/training/dms/dmd_manifest.csv \
    --validation-subject <subject_id> --device mps")